# GE-MolSG workflow

A single self-contained walkthrough of the GE-MolSG pipeline on a real
molecular surface (the thalidomide R enantiomer bundled under
`tests/data/`). Everything runs in process — no CLI calls or extra data files.

Stages covered:

1. Load a surface and build the field-augmented point cloud
2. Inspect the spectral intermediates: affinity matrix, graph Laplacian, eigensystem
3. Compute the per-vertex WKS descriptor
4. Generate descriptors for several surfaces (optionally in parallel)
5. Build a Bag-of-Features codebook and encode surfaces into fixed-length vectors

Each stage is an independent function, so you can stop at whichever output you need.

## Setup

Import the package and locate the bundled surface. Adjust `DATA` if you run this notebook from a different directory.

In [1]:
import warnings
from pathlib import Path

import numpy as np

import ge_molsg as gm

warnings.filterwarnings("ignore")  # silence a NumPy unpickling deprecation

# Path to the bundled thalidomide R surface (repo-relative).
DATA = Path("tests/data/thalidomide_R.npy")
if not DATA.exists():
    # fall back to a path relative to this notebook's examples/ location
    DATA = Path("../tests/data/thalidomide_R.npy")
print("surface file:", DATA.resolve())
print("ge_molsg version:", gm.__version__)

ModuleNotFoundError: No module named 'ge_molsg'

## 1. Load a surface

`load_surface_npy` reads the `[vertices, faces, charges]` object array into a `MolSurface`. The augmented point cloud appends the per-vertex ESP (scaled by `elec_weight`) as a fourth coordinate, so the neighbour search happens in a field-augmented Euclidean space.

In [ ]:
surface = gm.load_surface_npy(str(DATA), name="thalidomide_R")
print("name        :", surface.name)
print("n_vertices  :", surface.n_vertices)
print("faces       :", None if surface.faces is None else surface.faces.shape)
print("esp range   : [%.3f, %.3f]" % (surface.esp.min(), surface.esp.max()))

points = surface.augmented_points(elec_weight=0.3)
print("augmented   :", points.shape, "-> [x, y, z, esp * 0.3]")

## 2. Spectral intermediates

The descriptor is built from the bottom of the graph Laplacian's spectrum. Each step is exposed separately so you can inspect the affinity matrix, the Laplacian, and the eigensystem directly.

In [ ]:
N_NEIGHBORS = 100
N_COMPONENTS = 100

# 2a. Adaptive-bandwidth affinity matrix (sparse).
W = gm.compute_affinity(points, n_neighbors=N_NEIGHBORS)
print("affinity W  :", W.shape, "nnz =", W.nnz)

# 2b. Normalized graph Laplacian (sparse).
L = gm.graph_laplacian(W, laplacian_type="normalized")
print("laplacian L :", L.shape, "symmetric =", np.allclose((L - L.T).toarray(), 0, atol=1e-10))

# 2c. Bottom eigenpairs (trivial mode dropped).
eigenvalues, eigenvectors = gm.compute_eigensystem(L, n_components=N_COMPONENTS)
print("eigenvalues :", eigenvalues.shape, "ascending =", bool(np.all(np.diff(eigenvalues) >= -1e-9)))
print("eigenvectors:", eigenvectors.shape)
print("first 5 evals:", np.round(eigenvalues[:5], 5))

## 3. Per-vertex WKS descriptor

`wks` turns the eigensystem into a per-vertex Wave Kernel Signature. `compute_wks` does the whole chain (affinity → Laplacian → eigensystem → WKS) in one call from a config.

In [ ]:
# Directly from the eigensystem computed above:
desc_from_eig = gm.wks((eigenvalues, eigenvectors), evals=50)
print("from eigensystem:", desc_from_eig.shape)

# Or end-to-end from the surface via a config:
config = gm.GEMolSGConfig(n_neighbors=N_NEIGHBORS, n_components=N_COMPONENTS, evals=50)
descriptor = gm.compute_wks(surface, config)
print("from compute_wks:", descriptor.shape, "finite =", np.isfinite(descriptor).all())

## 4. Many surfaces

Real use involves many molecules. `compute_wks_batch` runs descriptor generation over a list of surfaces, optionally across processes (`n_jobs=-1` uses all CPUs), returning results in input order.

Here we synthesise a few extra surfaces by perturbing the real one so the example is self-contained; in practice you would load a directory of `.npy` files.

In [ ]:
def jitter(surf, seed):
    rng = np.random.default_rng(seed)
    v = surf.vertices + 0.02 * rng.normal(size=surf.vertices.shape)
    return gm.MolSurface(vertices=v, faces=surf.faces, esp=surf.esp, name=f"{surf.name}_{seed}")

surfaces = [surface] + [jitter(surface, s) for s in range(1, 4)]
print("surfaces:", [s.name for s in surfaces])

descriptors = gm.compute_wks_batch(surfaces, config, n_jobs=1, progress=True)
print("descriptor count:", len(descriptors))
print("shapes:", {d.shape for d in descriptors})

## 5. Codebook and Bag-of-Features

To get one fixed-length vector per molecule, pool the per-vertex descriptors, fit a codebook (MiniBatchKMeans), and encode each surface against it with a k-NN histogram.

In [ ]:
# Pool descriptors (optionally farthest-point-subsampled per molecule) and fit a codebook.
sampled_desc = gm.sample_descriptors(descriptors, n_per_mol=300)
pool = gm.build_descriptor_pool(sampled_desc)
codebook = gm.build_codebook(pool, n_codewords=128)
print("pool    :", pool.shape)
print("codebook:", codebook.shape)

# Encode each surface to a per-molecule BoF vector.
vectors = np.vstack([gm.knn_histogram(d, codebook, knn=3) for d in descriptors])
print("vectors :", vectors.shape, "rows sum to 1 =", np.allclose(vectors.sum(axis=1), 1.0))

## 6. Comparing molecules

With one vector per molecule, cosine similarity gives a simple comparison. The first surface is the real one; the rest are jittered copies, so they should be highly similar to it.

In [ ]:
def cosine(a, b):
    return float(a @ b / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-12))

ref = vectors[0]
for surf, vec in zip(surfaces, vectors):
    print(f"{surf.name:20s} cosine to reference = {cosine(ref, vec):.4f}")

## Convenience wrapper

The `GEMolSG` class composes the steps above behind `fit_codebook` / `transform` if you prefer a single object holding the config and codebook. It adds nothing the free functions above cannot do.

In [ ]:
model = gm.GEMolSG(config, n_codewords=128, sample_n_per_mol=300, n_jobs=1)
model.fit_codebook(surfaces)
vector = model.transform(surface)
print("single transform:", vector.shape)
print("batch transform :", model.transform_many(surfaces).shape)